# 🧬 EGFR Conditional Binder — Complex Structure & Interface Validation
## Anthropic × Adaptyv Protein Design Competition 2026 — Challenge 1
**User:** Saran Boddu ()
**Target:** EGFR Extracellular Domain III (Residues 310–481)
**Designs:** 13 De Novo Single-Chain pH-Conditional 3-Helix Bundles (69–70 aa)


In [ ]:
import torch
import os
import sys
import subprocess

print('=== GPU & SYSTEM ENVIRONMENT ===')
print('Python version:', sys.version.split()[0])
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device name:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 2), 'GB')
else:
    print('Running on CPU')

# Install biopython if needed
subprocess.run(['pip', 'install', '-q', 'biopython'], check=False)


In [ ]:
import json
import csv

# EGFR Domain III Target (human residues 310-481, 172 aa)
# Fully conserved with mouse EGFR at key anchor positions: D355, E352, E367, D392
EGFR_DOMAIN_III = (
    'CQGTSNKLTQLGTFEDHFLSLQRMFNNCEVVLGNLEITYVQRNYDLSFLKTIQEVAGYVLIA'
    'LNTVERIPLENLQIIRGNMYYENSYALAVLSNYDANKTGLKELPMRNLQEILHGAVRFSNNPA'
    'LCNVESIQWRDIVSSDFLSNMSMDFQNHLGSCQKC'
)

# 13 verified candidate designs (ESMFold passed, Cys=0, Novelty >= 3/4)
CANDIDATES = [
    {'rank': 1, 'name': 'EGFR-pH-HB-B02', 'seq': 'WYQAHHLKALLAEELANLKRLEEAQKRLEEQLKGSGSLAAALRRLEQALREQKLGSLEHHLRRLEQALK', 'his': 4},
    {'rank': 2, 'name': 'EGFR-pH-HB-B01', 'seq': 'FYNAHHLKALLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEHHLRRLEQALK', 'his': 4},
    {'rank': 3, 'name': 'EGFR-pH-HB-G01', 'seq': 'FYNAHHLKAHLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK', 'his': 4},
    {'rank': 4, 'name': 'EGFR-pH-HB-B03', 'seq': 'FYNAHHLKALLQQELAELKRLEQAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQHLK', 'his': 4},
    {'rank': 5, 'name': 'EGFR-pH-HB-D02', 'seq': 'WYNKHLARLHLLQQELAELKRLEEAQKRLEQALKGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK', 'his': 3},
    {'rank': 6, 'name': 'EGFR-pH-HB-A02', 'seq': 'WYQAHHLKALLAEELANLKRLEEAQKRLEEQLKGSGSLAAALRRLEQALREQKLGSLEAHLRRLEQALK', 'his': 3},
    {'rank': 7, 'name': 'EGFR-pH-HB-C01', 'seq': 'FYNRAHHLKALLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK', 'his': 3},
    {'rank': 8, 'name': 'EGFR-pH-HB-A05', 'seq': 'YWNAHHLKALLQQELAQMKRLEEAQKRLEQALKGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK', 'his': 3},
    {'rank': 9, 'name': 'EGFR-pH-HB-A03', 'seq': 'FYNHHALKALLQEELAQMKRLEEAQKRLEQALKGSGSLAAALRRLEQALREQKLGSLEAHLRRLEQALK', 'his': 3},
    {'rank': 10, 'name': 'EGFR-pH-HB-D01', 'seq': 'FYNKHLAKLHLLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK', 'his': 3},
    {'rank': 11, 'name': 'EGFR-pH-HB-A01', 'seq': 'FYNAHHLKALLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK', 'his': 3},
    {'rank': 12, 'name': 'EGFR-pH-HB-A04', 'seq': 'FYNAHHLRALLQQELAQLKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK', 'his': 3},
    {'rank': 13, 'name': 'EGFR-pH-HB-C03', 'seq': 'FYNKAHHLALLQQELAELKRLEEAQKRLEQALQGSGSLEAALRRLEQALREQKLGSLEAHLRRLEQALK', 'his': 3}
]

print(f'Loaded {len(CANDIDATES)} candidates. Target EGFR Domain III length: {len(EGFR_DOMAIN_III)} aa')


In [ ]:
import requests
import time
import numpy as np
from Bio import PDB
import os
import json

os.makedirs('/kaggle/working/complexes', exist_ok=True)
url = 'https://api.esmatlas.com/foldSequence/v1/pdb/'

parser = PDB.PDBParser(QUIET=True)
complex_results = []

print('=== PREDICTING BINDER-EGFR COMPLEX STRUCTURES ===')

for i, cand in enumerate(CANDIDATES, 1):
    name = cand['name']
    b_seq = cand['seq']
    linker = 'G' * 25
    complex_seq = b_seq + linker + EGFR_DOMAIN_III
    out_pdb = f'/kaggle/working/complexes/{name}_EGFR_complex.pdb'
    
    print(f'[{i}/{len(CANDIDATES)}] Folding {name} + EGFR Domain III ({len(complex_seq)} aa)...')
    
    try:
        resp = requests.post(url, data=complex_seq, timeout=180)
        if resp.status_code == 200 and 'ATOM' in resp.text:
            with open(out_pdb, 'w') as f:
                f.write(resp.text)
            print(f'  Saved complex structure to {out_pdb}')
            
            s = parser.get_structure(name, out_pdb)
            ca_atoms = [a for a in s.get_atoms() if a.name == 'CA']
            bfactors = [a.bfactor for a in ca_atoms]
            is_0_1 = max(bfactors) <= 1.0
            plddts = [b * 100.0 if is_0_1 else b for b in bfactors]
            
            binder_len = len(b_seq)
            binder_plddt = float(np.mean(plddts[:binder_len]))
            egfr_plddt = float(np.mean(plddts[binder_len + len(linker):]))
            complex_plddt = float(np.mean(plddts))
            
            complex_results.append({
                'rank': cand['rank'],
                'name': name,
                'binder_plddt': round(binder_plddt, 2),
                'egfr_plddt': round(egfr_plddt, 2),
                'complex_plddt': round(complex_plddt, 2),
                'pdb_path': out_pdb
            })
            print(f'  Metrics: Binder pLDDT={binder_plddt:.1f}, EGFR pLDDT={egfr_plddt:.1f}, Overall={complex_plddt:.1f}')
        else:
            print(f'  API returned code {resp.status_code}: {resp.text[:100]}')
    except Exception as e:
        print(f'  Error: {e}')
    time.sleep(1)

with open('/kaggle/working/complex_results.json', 'w') as f:
    json.dump(complex_results, f, indent=2)

print('Done folding complexes. Total generated:', len(complex_results))


In [ ]:
import pandas as pd

if complex_results:
    df = pd.DataFrame(complex_results)
    df.to_csv('/kaggle/working/egfr_complex_validation_summary.csv', index=False)
    print('=== COMPLEX VALIDATION SUMMARY ===')
    print(df[['rank', 'name', 'binder_plddt', 'egfr_plddt', 'complex_plddt']])
else:
    print('No complex results generated.')
